# Zambia VACS 2014 — PUD exploration

Public-use microdata live under `data/raw/Zambia Stata/`. **Male** and **Female** respondent files are **separate** (split-sample EAs); structure is almost identical (`id` ranges do not overlap between files).

This notebook: load → raw samples → **harmonized slot summary** (table below) → **§2** variable table → **`utils.checklist`** (**`type_and_width`** + TSV) → further use of `df`.

In [ ]:
from pathlib import Path
import sys

from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyreadstat
import re

_repo = next(
    (
        d
        for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (d / "utils" / "repo_paths.py").is_file() and (d / "data" / "raw").is_dir()
    ),
    None,
)
if _repo is None:
    raise RuntimeError("Cannot find repo root (expected utils/repo_paths.py and data/raw/).")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from utils.repo_paths import find_repo_root

ROOT = find_repo_root()
print("ROOT =", ROOT)

ZAMBIA_DIR = ROOT / "data" / "raw" / "Zambia Stata"
MALE_DTA = ZAMBIA_DIR / "ZAMBIA_VACS_2014_Male_PUD.dta"
FEMALE_DTA = ZAMBIA_DIR / "ZAMBIA_VACS_2014_Female_PUD.dta"

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

## 1. Load data

Primary frame: **`ZAMBIA_VACS_2014_Male_PUD.dta`**. The female file uses the same variable names for design/geo fields; row count differs (891 females, 928 males).

In [ ]:
DTA_PATH = MALE_DTA
if not DTA_PATH.is_file():
    raise FileNotFoundError(f"Expected:\n  {DTA_PATH}")

df, meta = pyreadstat.read_dta(DTA_PATH)
print(f"File: {DTA_PATH}")
print(f"Rows × columns: {df.shape[0]:,} × {df.shape[1]:,}")
if FEMALE_DTA.is_file():
    df_f, _ = pyreadstat.read_dta(FEMALE_DTA)
    print(f"Female PUD (for reference): {df_f.shape[0]:,} × {df_f.shape[1]:,}")
if getattr(meta, "file_label", None):
    print(f"Stata dataset label: {meta.file_label!r}")
df.head()

In [ ]:
df['Q2'].agg(['mean', 'median', 'std', 'min', 'max'])

### Raw row samples

Head / tail / fixed random sample, plus a **narrow** subset of IDs, geography, cluster, and interview date fields.

In [ ]:
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

print("--- head(8) — all columns ---")
display(df.head(8))

print("\n--- tail(4) ---")
display(df.tail(4))

print("\n--- sample(6, random_state=0) ---")
display(df.sample(6, random_state=0))

_core_cols = [
    c
    for c in [
        "id",
        "psu",
        "hh",
        "prov",
        "dist",
        "const",
        "ward",
        "csa",
        "nsel",
        "ntot",
        "HYR_VF",
        "HMTH_VF",
        "HDAY_VF",
        "VISIT_NF",
        "hcluster",
        "Q2"
    ]
    if c in df.columns
]
subset = df[_core_cols]
print(f"\n--- head(8) / sample — {len(_core_cols)} columns ---")
display(subset.head(8))
display(subset.sample(6, random_state=0))

### Harmonized codebook slots — Zambia 2014 (Male PUD; Female PUD parallel)

Use this table when filling your Excel draft. Every row includes **data source**, **variable name(s)**, and notes with **(a) data type** and **(b) digits / character width** (from the PUDs; verify analysis rules in `ZAMBIA_VACS_2014_DataUserGuide.pdf`).

**Shared data source:** `ZAMBIA_VACS_2014_Male_PUD.dta` and `ZAMBIA_VACS_2014_Female_PUD.dta` (same variables for design/geo/id fields; row counts differ).

| Slot | Data source | Variable(s) | Note |
|------|-------------|-------------|------|
| Respondent ID | Male + Female PUD above | `id` | **(a)** Type: `str` (characters are digits 0–9 only). **(b)** Width: 6 characters. Unique within each file; IDs do not overlap across Male vs Female PUD. |
| Household ID | Male + Female PUD above | `psu` + `hh` | **(a)** Type: `int` + `int` (EA ID + household number within EA). **(b)** Digits: `psu` is 1–3 in Male PUD, 2–3 in Female PUD; `hh` is 1–3 in both. Composite `psu`+`hh` is unique per row in each file. |
| Geo level 1 | Male + Female PUD above | `prov` | **(a)** Type: `int` (province code). **(b)** Digits: 1–2 (values 1–10 in both PUDs). |
| Geo level 2 (district code) | Male + Female PUD above | `dist` | **(a)** Type: `int` (district code). **(b)** Digits: 3–4 (Male ~101–1007; Female ~101–1006). **Not** the EA / Admin 2 slot for this project—see **`psu`**. |
| Geo level 2 / Admin 2 — EA | Male + Female PUD above | `psu` | **Enumeration area** (Stata *EA ID*). Same variable as **Cluster** below—cross-reference in Excel **notes** (`skills/memory.md`). |
| Cluster | Male + Female PUD above | `psu` | **(a)** Type: `int` (EA / PSU for `svy`). **(b)** Digits: same as `psu` in Household row. |
| Interview date | Male + Female PUD above | `HYR_VF` + `HMTH_VF` + `HDAY_VF` | **(a)** Type: three separate `int` fields (final visit year, month, day). **(b)** Digits: year 4; month 1–2 (e.g. 8–10 in extract); day 1–2 (e.g. 1–30 Male, 1–28 Female). |

In [ ]:
L = meta.column_names_to_labels or {}

def slot_summary(name, cols, df):
    """One slot: single column or composite (join with '+'; uniqueness on concatenated key)."""
    missing = [c for c in cols if c not in df.columns]
    if missing:
        return {"slot": name, "variable": "; ".join(cols), "note": f"MISSING: {missing}"}
    if len(cols) == 1:
        col = cols[0]
        s = df[col]
        lbl = L.get(col, "") or ""
        if pd.api.types.is_numeric_dtype(s):
            width = f"values min={int(s.min())}, max={int(s.max())}"
        else:
            lens = s.astype(str).str.len()
            width = f"{int(lens.min())}-{int(lens.max())} chars"
        uni = s.nunique(dropna=True)
        return {
            "slot": name,
            "variable": col,
            "stata_label": (lbl or "")[:80],
            "dtype": str(s.dtype),
            "n_distinct": int(uni),
            "n_rows": len(s),
            "unique_per_row": bool(uni == len(s) and s.notna().all()),
            "width_hint": width,
            "sample": list(s.dropna().head(5)),
        }
    parts = [df[c].astype(str) for c in cols]
    key = parts[0]
    for p in parts[1:]:
        key = key + "_" + p
    uni = key.nunique()
    labels_join = " | ".join((L.get(c, "") or "")[:40] for c in cols)
    return {
        "slot": name,
        "variable": " + ".join(cols),
        "stata_label": labels_join[:200],
        "dtype": "composite (" + ", ".join(str(df[c].dtype) for c in cols) + ")",
        "n_distinct": int(uni),
        "n_rows": len(df),
        "unique_per_row": bool(uni == len(df) and not key.duplicated().any()),
        "width_hint": "per-component ranges below",
        "sample": df[cols].head(5).to_dict(orient="records"),
    }


PIPELINE = [
    ("1. Respondent ID", ["id"]),
    ("2. Household ID", ["psu", "hh"]),
    ("3. Geo level 1 (region/province)", ["prov"]),
    ("4. District code (not EA)", ["dist"]),
    ("5. Admin 2 / EA & cluster (psu)", ["psu"]),
    ("6. Interview date", ["HYR_VF", "HMTH_VF", "HDAY_VF"]),
]

print("Data source:", DTA_PATH.name)
rows = [slot_summary(name, cols, df) for name, cols in PIPELINE]
pipe_df = pd.DataFrame(rows)
display(pipe_df.drop(columns=["sample"], errors="ignore"))

print("\nSamples:")
for r in rows:
    print(r["slot"], "→", r.get("sample"))

print("\n--- Frequency: prov, dist (top 12) ---")
display(df["prov"].value_counts().head(12))
display(df["dist"].value_counts().head(12))

print("\n--- Cross-check: female file `id` overlap with male (expect 0) ---")
if FEMALE_DTA.is_file():
    df_f, _ = pyreadstat.read_dta(FEMALE_DTA)
    ov = set(df["id"].astype(str)) & set(df_f["id"].astype(str))
    print("overlap count:", len(ov))

## 2. What’s in this file?

Stata labels, dtypes, missingness (first 40 variables + highest missing %). Run the **checklist** cell next for **`type_and_width`**, **`suggested_layout`**, and TSV (`utils.checklist`); edit **`CANDIDATES`** as needed.

In [ ]:
name_to_label = dict(meta.column_names_to_labels) if meta.column_names_to_labels else {}
var_table = pd.DataFrame({
    "column": df.columns,
    "stata_label": [name_to_label.get(c, "") or "" for c in df.columns],
    "dtype": df.dtypes.astype(str).values,
    "missing_n": df.isna().sum().values,
    "missing_pct": (100 * df.isna().mean()).round(2),
})

print(f"Variables: {len(df.columns):,}  |  Observations: {len(df):,}")
print(f"Embedded Stata value-label maps: {len(meta.value_labels or {})}")
display(var_table.head(40))
display(
    var_table.sort_values("missing_pct", ascending=False)
    .head(15)
    .reset_index(drop=True)
)
df.info(max_cols=20)

### Harmonized geography / ID checklist (`utils.checklist`)

**Admin 2 = enumeration areas (EAs):** Stata labels **`psu`** as *EA ID*—use **`psu`** for **GeoLevel2 / Admin 2** (not **`dist`**, which is the **district** code layer). **`psu`** is also the **cluster** for design—duplicate Excel rows or cross-reference in **notes** (`skills/memory.md`).

This cell summarizes **`df`** (Male PUD loaded above). Female PUD uses the **same** Stata names for these fields—re-run after loading female if needed.

Edit **`CANDIDATES`** after §2 if you change the column set.


In [ ]:
# You choose candidates after §2 EDA; utils summarize columns present in `df`.
from utils.checklist import build_checklist_df, checklist_to_tsv

CANDIDATES = [
    ("Admin 1 (province)", ["prov"]),
    ("District code (~1.5)", ["dist"]),
    ("Admin 2 — enumeration area (EA)", ["psu"]),
    ("Cluster (svy; same EA as psu)", ["psu"]),
    ("Household # (within EA)", ["hh"]),
    ("Respondent ID", ["id"]),
    ("Finer geo / optional", ["ward", "const", "csa"]),
    ("Roster / selection context", ["nsel", "ntot"]),
    ("Split-sample / design", ["hcluster"]),
    ("Field / visit date parts", ["HYR_VF", "HMTH_VF", "HDAY_VF", "VISIT_NF"]),
]

_labels = meta.column_names_to_labels or {}
checklist_df = build_checklist_df(df, CANDIDATES, column_labels=_labels)

with pd.option_context("display.max_colwidth", 100, "display.width", 220):
    display(checklist_df)

print("\n--- TSV (copy for Excel / codebook) ---\n")
print(checklist_to_tsv(checklist_df))
